In [1]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Gene Exp

In [2]:
drug_response = pd.read_csv("../gdsc2_data/drugAct.csv", index_col=0)
gene_exp = pd.concat(
    [
        pd.read_csv("../gdsc2_data/gene_exp_part1.csv.gz", index_col=0).T,
        pd.read_csv("../gdsc2_data/gene_exp_part2.csv.gz", index_col=0).T,
    ],
    axis=1,
).dropna()
cells = sorted(
    set(drug_response.columns)
    & set(gene_exp.index)
    & set(pd.read_csv("../gdsc2_data/mut.csv", index_col=0).T.index)
)

In [3]:
# def get_smiles_from_compound_name(compound_name, max_retries=5, delay=5):
#     # First try to get SMILES from local file
#     try:
#         df = pd.read_csv(
#             "../Figs/nsc_cid_smiles_class_name.csv", usecols=["NAME", "SMILES"]
#         )
#         match = df[df["NAME"].str.lower() == compound_name.lower()]
#         if not match.empty:
#             return match.iloc[0]["SMILES"]
#     except Exception as e:
#         print(f"Error reading local SMILES data: {e}")

#     # If not found locally, try PubChem API
#     url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{compound_name}/property/CanonicalSMILES/JSON"

#     for attempt in range(max_retries):
#         try:
#             response = requests.get(url)
#             response.raise_for_status()
#             data = response.json()
#             smiles = (
#                 data.get("PropertyTable", {})
#                 .get("Properties", [{}])[0]
#                 .get("CanonicalSMILES")
#             )
#             return smiles
#         except requests.RequestException as e:
#             if response.status_code == 503 or "PUGREST.ServerBusy" in str(e):
#                 print(
#                     f"Server busy. Retrying in {delay} seconds... (Attempt {attempt + 1}/{max_retries})"
#                 )
#                 time.sleep(delay)
#                 delay *= 2  # Exponential backoff
#             else:
#                 print(f"Error retrieving SMILES for compound name {compound_name}: {e}")
#                 return None

#     print(f"Failed to retrieve SMILES after {max_retries} attempts")
#     return None


# def process_compound(compound_name):
#     return [get_smiles_from_compound_name(compound_name), compound_name]


# def get_smiles_parallel(compound_names, max_workers=10):
#     results = []
#     with ThreadPoolExecutor(max_workers=max_workers) as executor:
#         futures = [executor.submit(process_compound, name) for name in compound_names]
#         for future in tqdm(
#             as_completed(futures),
#             total=len(compound_names),
#             desc="Processing compounds",
#         ):
#             results.append(future.result())
#     return results

# Drug Response

In [4]:
# compound_names = drug_response.index.tolist()
# SMILES = get_smiles_parallel(compound_names)

In [5]:
# SMILES = pd.DataFrame(SMILES)
# SMILES

In [6]:
# SMILES[SMILES[0].isna()].shape

In [7]:
# sorted(SMILES[SMILES[0].isna()][1])

In [8]:
# missing = [
#     "123138",
#     "123829",
#     "150412",
#     "50869",
#     "615590",
#     "630600",
#     "667880",
#     "720427",
#     "729189",
#     "741909",
#     "743380",
#     "765771",
#     "776928",
#     "Acetalax",
#     "Acetalax",
#     "BDF00022089a",
#     "BDILV000379a",
#     "BDOCA000347a",
#     "BDP-00009066",
#     "BPD-00008900",
#     "Bleomycin",
#     "CDK9",
#     "CDK9",
#     "CT7033-2",
#     "Dactinomycin",
#     "Dactinomycin",
#     "Docetaxel",
#     "Docetaxel",
#     "ERK_2440",
#     "ERK_6604",
#     "Eg5_9814",
#     "Fulvestrant",
#     "Fulvestrant",
#     "GSK-LSD1-2HCl ",
#     "GSK2256098C",
#     "GSK2276186C",
#     "GSK2830371A",
#     "GSK3337463A",
#     "GSK343_1627",
#     "GSK343_2037",
#     "GSK626616AC",
#     "HKMTI-1-005",
#     "IAP",
#     "IGF1R",
#     "IRAK4",
#     "JAK1",
#     "JAK",
#     "KRAS (G12C) Inhibitor-12",
#     "LMB_AB1",
#     "LMB_AB2",
#     "LMB_AB3",
#     "N25720-51-A1",
#     "N27922-53-1",
#     "N29087-69-1",
#     "N30652-18-1",
#     "Nutlin-3a (-)",
#     "Oxaliplatin",
#     "Oxaliplatin",
#     "PAK",
#     "PBD-288",
#     "Picolinici-acid",
#     "Selumetinib",
#     "Selumetinib",
#     "TAF1",
#     "THR-101",
#     "THR-102",
#     "THR-103",
#     "ULK1_4989",
#     "Ulixertinib",
#     "Ulixertinib",
#     "Uprosertib",
#     "Uprosertib",
#     "VSP34",
#     "VTP-A",
#     "VTP-B",
#     "ascorbate",
# ]

In [9]:
# missing_SMILES = get_smiles_parallel(missing)

In [10]:
# k = pd.DataFrame([missing, sorted(SMILES[SMILES[0].isna()][1])]).T
# k.columns = [1, 2]

In [11]:
# tmp = pd.DataFrame(missing_SMILES).merge(k).drop_duplicates().drop(1, axis=1).dropna()
# tmp.columns = ["SMILES", "drugs"]
# SMILES = SMILES.dropna()
# SMILES.columns = ["SMILES", "drugs"]
# SMILES = pd.concat([tmp, SMILES])

In [12]:
# SMILES

In [13]:
# SMILES.to_csv('gdsc2_drug2smiles.csv')

In [14]:
SMILES = pd.read_csv("gdsc2_drug2smiles.csv", index_col=0)
SMILES

,SMILES,drugs
12,CC1=C(N=C(N=C1N)C(CC(=O)N)NCC(C(=O)N)N)C(=O)NC...,Bleomycin (50 uM)
18,CC1C(C(=O)NC(C(=O)N2CCCC2C(=O)N(CC(=O)N(C(C(=O...,Dactinomycin_1811
19,CC1C(C(=O)NC(C(=O)N2CCCC2C(=O)N(CC(=O)N(C(C(=O...,Dactinomycin_1911
20,CC1=C2C(C(=O)C3(C(CC4C(C3C(C(C2(C)C)(CC1OC(=O)...,Docetaxel_1007
21,CC1=C2C(C(=O)C3(C(CC4C(C3C(C(C2(C)C)(CC1OC(=O)...,Docetaxel_1819
...,...,...
285,CC(=O)NC(CS)C(=O)O,N-acetyl cysteine
289,COC1=C(C=C2C(=C1)C(=NC(=N2)N3CCCC3)NCCCCCN4CCC...,UNC0379
291,C1=CC(=CC=C1C2=CNN=C2)C(CN)(C3=CC=C(C=C3)Cl)O,AT13148
292,CC1=C(C=C(C=N1)Cl)NCC2=CC=C(S2)C(=O)NC(CC3CCCC...,GSK2830371


In [15]:
gene_exp = gene_exp.loc[cells]
gene_exp

Gene,A1BG,A1CF,A2M,A2ML1,A3GALT2,A3GALT2P,A4GALT,A4GNT,AAAS,AACS,...,ZW10,ZWILCH,ZWINT,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
22RV1,3.537942,6.364651,5.332441,2.923030,2.815383,2.815383,3.241125,3.262633,4.722157,4.942126,...,8.378382,7.526319,9.274166,3.785757,5.201929,2.742575,4.635015,3.940190,4.658663,8.088672
23132-87,3.370950,6.284884,3.485675,2.831562,2.913369,2.913369,3.313028,3.096527,4.873621,4.213177,...,8.586287,6.526713,8.021952,3.520584,4.957371,2.636933,4.350120,4.395806,4.598583,7.951146
42-MG-BA,6.016743,3.191894,3.249480,2.808959,2.626195,2.626195,3.063389,3.356853,4.919962,4.978849,...,8.574633,7.274847,7.683935,3.473062,5.237112,2.759740,5.517990,4.391955,4.157708,8.944875
451Lu,5.725238,2.971953,10.768500,2.928547,2.750283,2.750283,3.367116,3.162653,5.144177,4.242312,...,7.805859,7.942236,9.385336,3.292793,5.530937,2.582158,5.087147,5.197606,4.922167,7.470603
5637,2.927335,2.892365,3.181651,2.926549,2.677943,2.677943,4.295357,3.205598,5.249042,4.495021,...,8.498743,7.057572,9.261486,3.492602,4.813163,2.558705,4.580977,4.810975,4.371501,7.974367
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YT,3.143593,3.779792,5.095163,2.879333,2.749091,2.749091,3.186225,3.359589,4.511721,4.241912,...,8.467307,8.196766,9.205680,3.515323,5.152341,2.472749,5.046783,6.016707,5.168183,7.063949
ZR-75-30,5.594177,3.032126,3.562195,3.108549,2.767673,2.767673,3.611381,3.247737,4.547910,4.423134,...,7.725739,7.398583,8.781244,3.574957,5.301549,3.045222,4.952083,4.604161,4.472996,7.675603
huH-1,4.177894,10.272718,5.050040,2.904163,3.162503,3.162503,3.205621,3.332665,5.467095,3.694510,...,8.486763,6.799690,8.755107,3.602078,5.365176,2.990428,4.844390,4.089650,5.042040,8.574266
no-10,4.106684,3.229351,3.070295,2.793178,2.806316,2.806316,3.203713,3.365893,4.312271,4.227046,...,8.861172,7.475805,8.647136,3.569171,5.473651,2.534410,4.967557,4.798076,4.447893,8.113379


In [16]:
drug_response = drug_response.loc[sorted(SMILES.drugs), cells]
drug_response

,22RV1,23132-87,42-MG-BA,451Lu,5637,639-V,647-V,697,769-P,786-0,...,WSU-DLCL2,WSU-NHL,YAPC,YH-13,YKG-1,YT,ZR-75-30,huH-1,no-10,no-11
5-Fluorouracil,-1.007383,-0.784918,-1.504194,-1.703398,-1.354900,-1.987515,-2.477538,-0.978509,-1.361415,-1.754028,...,-1.570807,-0.654756,-2.815973,-2.167938,-1.897658,-0.395993,-3.121705,-2.718147,-2.158567,-2.354397
5-azacytidine,-1.386340,-0.882578,-1.740183,NaN,-1.642768,-0.769647,-1.478863,0.076926,-1.204752,-1.175576,...,-0.196915,-0.318916,-1.863731,-1.153293,NaN,-0.719780,-2.407082,-1.032133,NaN,NaN
A-366,-2.034810,-1.723082,-2.213116,NaN,-2.101929,-2.346732,-2.175772,-1.446605,-1.556993,-2.153308,...,-0.698646,-1.329787,-2.031633,-1.847300,NaN,-1.743936,-2.828028,-2.721790,NaN,NaN
ABT737,-1.731862,-1.285725,-1.311469,-1.176737,-1.118458,-1.923063,-1.495390,1.067689,-0.742262,-1.414749,...,0.402131,0.039985,-2.132064,-1.553697,-1.613575,-1.228735,-1.624036,-1.200161,-1.349079,-1.262084
AGI-5198,-2.716713,-2.175316,-2.030678,-2.144917,-1.930115,-1.715597,-2.176160,-1.681225,-2.522207,-1.719376,...,-1.360011,-1.364146,-2.942588,-2.093594,-1.516215,-1.454616,-2.787429,-2.501823,-2.208319,-2.704476
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZM447439,-2.461575,-1.853879,-0.752992,-2.146206,-0.422651,-1.395663,-0.798400,0.329506,-1.827889,-0.986953,...,-0.552339,0.193145,-2.743585,-1.736393,-0.726710,-0.566273,-1.962101,-0.955661,-1.499584,-1.896599
Zoledronate,-1.787369,-1.517874,-1.544003,NaN,-1.748420,-1.364055,-1.718075,-1.009534,-1.717543,-1.310182,...,NaN,-1.333803,-2.311593,-1.774683,NaN,NaN,-2.227486,-1.900266,NaN,NaN
alpha-lipoic acid,-3.448798,-3.495767,-3.460486,NaN,-3.172817,-3.338942,-3.288135,-2.957743,-3.448755,-3.419779,...,-2.616572,-3.143894,-3.982522,-3.380398,NaN,-3.137593,-3.831928,-3.665569,NaN,NaN
ascorbate (vitamin C),-5.195668,-4.109346,-4.177796,NaN,-4.133578,-4.659277,-4.043361,-3.548887,-4.462722,-4.266191,...,-2.947327,-4.362139,-5.091152,-4.231438,NaN,-3.915763,-5.227466,-4.875701,NaN,NaN


In [17]:
df = pd.DataFrame()
for i in drug_response.columns:
    tmp = pd.DataFrame(drug_response[i]).reset_index().dropna()
    tmp["cell"] = [i] * len(tmp)
    tmp.columns = ["Drug", "Value", "Cell"]
    tmp = tmp[["Drug", "Cell", "Value"]]
    df = pd.concat([df, tmp])
df

,Drug,Cell,Value
0,5-Fluorouracil,22RV1,-1.007383
1,5-azacytidine,22RV1,-1.386340
2,A-366,22RV1,-2.034810
3,ABT737,22RV1,-1.731862
4,AGI-5198,22RV1,-2.716713
...,...,...,...
231,Wee1 Inhibitor,no-11,-0.988306
232,Wnt-C59,no-11,-2.806986
233,XAV939,no-11,-1.673225
234,YK-4-279,no-11,-1.967464


# Zero shot prediction

In [18]:
unique_drugs = df["Drug"].unique()
unique_cells = df["Cell"].unique()

# Split drugs and cell lines into training, validation, and test sets
train_drugs, test_val_drugs = train_test_split(
    unique_drugs, test_size=0.5, random_state=42
)
val_drugs, test_drugs = train_test_split(test_val_drugs, test_size=0.5, random_state=42)

train_cells, test_val_cells = train_test_split(
    unique_cells, test_size=0.55, random_state=42
)
val_cells, test_cells = train_test_split(test_val_cells, test_size=0.5, random_state=42)

# Split the dataset
train_df = df[(df["Drug"].isin(train_drugs)) & (df["Cell"].isin(train_cells))]
val_df = df[(df["Drug"].isin(val_drugs)) & (df["Cell"].isin(val_cells))]
test_df = df[(df["Drug"].isin(test_drugs)) & (df["Cell"].isin(test_cells))]


# Function to balance label distribution
def balance_labels(df, threshold=0.5):
    positive = df[df["Value"] > 0]
    negative = df[df["Value"] <= 0]
    min_count = min(len(positive), len(negative))
    balanced_positive = positive.sample(min_count, random_state=42)
    balanced_negative = negative.sample(min_count, random_state=42)
    return pd.concat([balanced_positive, balanced_negative])


# Balance label distribution across all sets
train_df = balance_labels(train_df)
val_df = balance_labels(val_df)
test_df = balance_labels(test_df)

# Separate features and labels
X_train = train_df[["Drug", "Cell"]]
y_train = np.array(train_df["Value"] > 0, dtype=float)

X_val = val_df[["Drug", "Cell"]]
y_val = np.array(val_df["Value"] > 0, dtype=float)

X_test = test_df[["Drug", "Cell"]]
y_test = np.array(test_df["Value"] > 0, dtype=float)

# Calculate total samples
total_samples = len(X_train) + len(X_val) + len(X_test)


# Function to calculate and format label ratios
def get_label_ratio(y):
    unique, counts = np.unique(y, return_counts=True)
    total = len(y)
    ratio_str = ", ".join(
        [f"Label {label}: {count/total:.2%}" for label, count in zip(unique, counts)]
    )
    return ratio_str


# Display results with percentages and label ratios
print("Train:")
print(X_train.shape, y_train.shape)
print(f"Percentage: {len(X_train)/total_samples:.2%}")
print(f"Label Ratio: {get_label_ratio(y_train)}")

print("\nValidation:")
print(X_val.shape, y_val.shape)
print(f"Percentage: {len(X_val)/total_samples:.2%}")
print(f"Label Ratio: {get_label_ratio(y_val)}")

print("\nTest:")
print(X_test.shape, y_test.shape)
print(f"Percentage: {len(X_test)/total_samples:.2%}")
print(f"Label Ratio: {get_label_ratio(y_test)}")

print(f"\nTotal samples: {total_samples}")
print(
    f"Ratio (Train:Validation:Test): {len(X_train):.0f}:{len(X_val):.0f}:{len(X_test):.0f}"
)
print(
    f"Overall Label Ratio: {get_label_ratio(np.concatenate([y_train, y_val, y_test]))}"
)

X_train.to_csv("../gdsc2_data/train.csv", index=False)
X_test.to_csv("../gdsc2_data/test.csv", index=False)
X_val.to_csv("../gdsc2_data/val.csv", index=False)

np.save("../gdsc2_data/train_labels.npy", y_train)
np.save("../gdsc2_data/test_labels.npy", y_test)
np.save("../gdsc2_data/val_labels.npy", y_val)

Train:
(16174, 2) (16174,)
Percentage: 73.01%
Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%

Validation:
(2484, 2) (2484,)
Percentage: 11.21%
Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%

Test:
(3496, 2) (3496,)
Percentage: 15.78%
Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%

Total samples: 22154
Ratio (Train:Validation:Test): 16174:2484:3496
Overall Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%


In [19]:
drug_response = drug_response.fillna(0)
drug_response

,22RV1,23132-87,42-MG-BA,451Lu,5637,639-V,647-V,697,769-P,786-0,...,WSU-DLCL2,WSU-NHL,YAPC,YH-13,YKG-1,YT,ZR-75-30,huH-1,no-10,no-11
5-Fluorouracil,-1.007383,-0.784918,-1.504194,-1.703398,-1.354900,-1.987515,-2.477538,-0.978509,-1.361415,-1.754028,...,-1.570807,-0.654756,-2.815973,-2.167938,-1.897658,-0.395993,-3.121705,-2.718147,-2.158567,-2.354397
5-azacytidine,-1.386340,-0.882578,-1.740183,0.000000,-1.642768,-0.769647,-1.478863,0.076926,-1.204752,-1.175576,...,-0.196915,-0.318916,-1.863731,-1.153293,0.000000,-0.719780,-2.407082,-1.032133,0.000000,0.000000
A-366,-2.034810,-1.723082,-2.213116,0.000000,-2.101929,-2.346732,-2.175772,-1.446605,-1.556993,-2.153308,...,-0.698646,-1.329787,-2.031633,-1.847300,0.000000,-1.743936,-2.828028,-2.721790,0.000000,0.000000
ABT737,-1.731862,-1.285725,-1.311469,-1.176737,-1.118458,-1.923063,-1.495390,1.067689,-0.742262,-1.414749,...,0.402131,0.039985,-2.132064,-1.553697,-1.613575,-1.228735,-1.624036,-1.200161,-1.349079,-1.262084
AGI-5198,-2.716713,-2.175316,-2.030678,-2.144917,-1.930115,-1.715597,-2.176160,-1.681225,-2.522207,-1.719376,...,-1.360011,-1.364146,-2.942588,-2.093594,-1.516215,-1.454616,-2.787429,-2.501823,-2.208319,-2.704476
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZM447439,-2.461575,-1.853879,-0.752992,-2.146206,-0.422651,-1.395663,-0.798400,0.329506,-1.827889,-0.986953,...,-0.552339,0.193145,-2.743585,-1.736393,-0.726710,-0.566273,-1.962101,-0.955661,-1.499584,-1.896599
Zoledronate,-1.787369,-1.517874,-1.544003,0.000000,-1.748420,-1.364055,-1.718075,-1.009534,-1.717543,-1.310182,...,0.000000,-1.333803,-2.311593,-1.774683,0.000000,0.000000,-2.227486,-1.900266,0.000000,0.000000
alpha-lipoic acid,-3.448798,-3.495767,-3.460486,0.000000,-3.172817,-3.338942,-3.288135,-2.957743,-3.448755,-3.419779,...,-2.616572,-3.143894,-3.982522,-3.380398,0.000000,-3.137593,-3.831928,-3.665569,0.000000,0.000000
ascorbate (vitamin C),-5.195668,-4.109346,-4.177796,0.000000,-4.133578,-4.659277,-4.043361,-3.548887,-4.462722,-4.266191,...,-2.947327,-4.362139,-5.091152,-4.231438,0.000000,-3.915763,-5.227466,-4.875701,0.000000,0.000000


# Masking

In [20]:
# Validation Mask
for _, row in X_val.iterrows():
    drug_response.loc[row["Drug"], row["Cell"]] = 0

# Test Mask
for _, row in X_test.iterrows():
    drug_response.loc[row["Drug"], row["Cell"]] = 0

# DTI

In [21]:
dti = pd.read_csv("../data/full_table.csv")
dti = dti.dropna(subset="Drug Name").reset_index(drop=True)
dti.head()

,Drug Name,DrugBank ID,PubChem CID,PubChem SID,SMILES,Gene,NSC
0,Lepirudin,DB00001,NaN,46507011.0,NaN,F2,NaN
1,Cetuximab,DB00002,NaN,46507042.0,NaN,EGFR,NaN
2,Cetuximab,DB00002,NaN,46507042.0,NaN,FCGR3B,NaN
3,Cetuximab,DB00002,NaN,46507042.0,NaN,C1QA,NaN
4,Cetuximab,DB00002,NaN,46507042.0,NaN,C1QB,NaN


In [22]:
dti = dti[dti["Drug Name"].isin(drug_response.index)]
dti.head()

,Drug Name,DrugBank ID,PubChem CID,PubChem SID,SMILES,Gene,NSC
1061,Bortezomib,DB00188,387447.0,46508736.0,CC(C)C[C@H](NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)C1=...,PSMB5,681239.0
1062,Bortezomib,DB00188,387447.0,46508736.0,CC(C)C[C@H](NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)C1=...,PSMB1,681239.0
1454,Bleomycin,DB00290,5360373.0,46509116.0,[H][C@](C)(NC(=O)[C@@]([H])(NC(=O)C1=NC(=NC(N)...,LIG1,NaN
1455,Bleomycin,DB00290,5360373.0,46509116.0,[H][C@](C)(NC(=O)[C@@]([H])(NC(=O)C1=NC(=NC(N)...,LIG3,NaN
1555,Gefitinib,DB00317,123631.0,46508649.0,COC1=C(OCCCN2CCOCC2)C=C2C(NC3=CC(Cl)=C(F)C=C3)...,EGFR,715055.0


In [23]:
print("unique drugs:", len(set(dti["Drug Name"])))
print("unique genes:", len(set(dti.Gene)))

unique drugs: 69
unique genes: 157


In [24]:
len(set(drug_response.index) & set(dti["Drug Name"]))

69

In [25]:
len(set(gene_exp.columns) & set(dti.Gene))

153

## All drugs are in drug response.

In [26]:
dti = dti[dti.Gene.isin(set(gene_exp.columns) & set(dti.Gene))]
dti

,Drug Name,DrugBank ID,PubChem CID,PubChem SID,SMILES,Gene,NSC
1061,Bortezomib,DB00188,387447.0,46508736.0,CC(C)C[C@H](NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)C1=...,PSMB5,681239.0
1062,Bortezomib,DB00188,387447.0,46508736.0,CC(C)C[C@H](NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)C1=...,PSMB1,681239.0
1454,Bleomycin,DB00290,5360373.0,46509116.0,[H][C@](C)(NC(=O)[C@@]([H])(NC(=O)C1=NC(=NC(N)...,LIG1,NaN
1455,Bleomycin,DB00290,5360373.0,46509116.0,[H][C@](C)(NC(=O)[C@@]([H])(NC(=O)C1=NC(=NC(N)...,LIG3,NaN
1555,Gefitinib,DB00317,123631.0,46508649.0,COC1=C(OCCCN2CCOCC2)C=C2C(NC3=CC(Cl)=C(F)C=C3)...,EGFR,715055.0
...,...,...,...,...,...,...,...
19411,Foretinib,DB12307,42642645.0,347828572.0,COC1=C(OCCCN2CCOCC2)C=C2N=CC=C(OC3=CC=C(NC(=O)...,KDR,NaN
19413,Navitoclax,DB12340,24978538.0,347828599.0,[H][C@@](CCN1CCOCC1)(CSC1=CC=CC=C1)NC1=C(C=C(C...,BCL2,NaN
19414,Navitoclax,DB12340,24978538.0,347828599.0,[H][C@@](CCN1CCOCC1)(CSC1=CC=CC=C1)NC1=C(C=C(C...,BCL2L2,NaN
19415,Navitoclax,DB12340,24978538.0,347828599.0,[H][C@@](CCN1CCOCC1)(CSC1=CC=CC=C1)NC1=C(C=C(C...,BAD,NaN


# Selected highly variable genes

In [27]:
variance = gene_exp.std()
variance = variance.sort_values(ascending=False)
variance = pd.DataFrame(variance > np.percentile(variance, 90))
variance = list(variance[variance[0]].index)
len(variance)

1957

In [28]:
print("DTI unique genes: ", len(set(dti["Gene"])))
print("Top 90% variable genes: ", len(variance))
print("Total: ", len(set(variance) | (set(dti["Gene"]))))

DTI unique genes:  153
Top 90% variable genes:  1957
Total:  2089


# Preprocessed data dims

In [29]:
genes = sorted(list(set(variance) | (set(dti["Gene"]))))
gene_exp = gene_exp[genes]
gene_exp.shape

(910, 2089)

In [30]:
drug_response.shape

(240, 910)

# Normalize

In [31]:
gene_norm_cell = pd.DataFrame(
    StandardScaler().fit_transform(gene_exp),
    index=gene_exp.index,
    columns=gene_exp.columns,
)
gene_norm_cell

Gene,A2M,AAED1,ABCC1,ABCC3,ABCG1,ABHD17C,ABI3BP,ABL1,ABL2,ABLIM1,...,ZNF462,ZNF467,ZNF608,ZNF667-AS1,ZNF711,ZNF726,ZNF83,ZNF853,ZP3,ZSCAN31
22RV1,0.796411,-0.369331,-1.110903,-0.891672,-0.115062,0.353202,-0.574455,-0.385519,-0.854777,1.299575,...,-0.439929,-0.220893,-0.573352,0.923150,-0.030843,2.428089,0.716680,-0.946885,0.761208,0.088496
23132-87,-0.293087,0.435471,-0.987879,1.794547,2.448392,1.698704,-0.472722,-0.915705,-0.610633,1.831797,...,-1.157649,-0.152769,-0.831940,-0.771907,-0.929187,-0.796710,0.243376,-0.795623,1.237344,0.055365
42-MG-BA,-0.432430,0.313401,-0.626544,0.165835,-0.082646,-0.648717,-0.018534,0.903484,-0.216203,-1.652219,...,0.573886,-0.550130,0.967479,1.953295,0.809277,-0.590745,0.641629,0.100012,-1.278947,3.115442
451Lu,4.003412,-0.404608,-0.982501,-0.980493,-0.826507,0.259728,-0.398605,-1.238596,0.897031,-1.655669,...,0.255148,-0.978945,0.753573,-0.762227,-0.897595,-0.098917,-2.456974,-0.426107,-0.128830,-1.095479
5637,-0.472446,-0.555320,-0.085436,0.501026,-0.010092,0.004307,-0.387135,1.489354,0.290438,0.333228,...,1.644298,-0.863282,0.696653,-0.455103,0.174405,0.505963,-0.514894,1.634775,2.238949,0.163118
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YT,0.656429,-2.887004,-0.014616,-0.789311,-0.810201,0.127920,1.241993,-1.756988,-1.330044,0.372338,...,-1.362234,-0.829529,-1.214149,-0.784117,-0.603372,0.085387,0.687285,-0.634962,-0.973271,-1.277350
ZR-75-30,-0.247944,-0.974867,-0.531329,1.473019,2.176601,1.217080,-0.570480,-0.937799,0.331600,0.340156,...,-0.509508,1.770099,-0.933673,-0.753814,-0.917013,-0.590101,0.585016,0.306560,0.822555,-0.133459
huH-1,0.629809,0.953254,-1.023793,1.878570,-0.944805,-0.640888,-0.551479,0.294767,-1.718769,0.933714,...,-0.930349,-1.077466,-1.110137,0.453622,1.281734,1.702313,-1.216894,-1.017142,0.128524,2.393384
no-10,-0.538141,0.684416,-1.467586,0.464762,-0.887263,-0.535091,4.124889,-0.308275,1.087539,-0.270280,...,0.089058,-1.028840,0.360735,0.813918,0.812956,0.021504,0.825969,-0.430435,-1.434766,-0.566728


In [32]:
gene_norm_gene = pd.DataFrame(
    StandardScaler().fit_transform(gene_exp.T.values),
    index=gene_exp.columns,
    columns=gene_exp.index,
).T
gene_norm_gene

Gene,A2M,AAED1,ABCC1,ABCC3,ABCG1,ABHD17C,ABI3BP,ABL1,ABL2,ABLIM1,...,ZNF462,ZNF467,ZNF608,ZNF667-AS1,ZNF711,ZNF726,ZNF83,ZNF853,ZP3,ZSCAN31
22RV1,0.080014,0.392435,0.156115,-0.947237,-0.451814,0.949798,-1.049032,0.019321,-0.178059,1.522073,...,0.192754,-0.238140,-0.576581,0.789201,-0.383632,1.285661,1.576466,-1.041030,0.587504,-0.300608
23132-87,-0.748409,0.719625,0.107463,0.931301,0.973187,1.596991,-0.957493,-0.186801,-0.162373,1.667848,...,-0.348060,-0.239326,-0.734581,-0.896905,-1.003115,-0.620402,1.014199,-0.924472,0.726779,-0.357951
42-MG-BA,-0.933873,0.526903,0.090696,-0.300659,-0.557878,0.078899,-0.778039,0.163928,-0.163709,-0.756887,...,0.619961,-0.595597,0.151560,1.435193,0.020052,-0.608311,1.162249,-0.510463,-0.752064,1.135818
451Lu,2.221234,0.245316,0.083363,-1.026329,-0.917148,0.719120,-0.950714,-0.300032,0.226129,-0.700215,...,0.527204,-0.812486,0.124618,-0.925504,-1.020989,-0.278502,-1.003107,-0.747662,-0.052329,-1.011993
5637,-1.091667,-0.026389,0.141167,-0.178419,-0.635120,0.371858,-1.125534,0.219139,-0.130896,0.452826,...,1.219209,-0.919710,-0.096885,-0.826343,-0.497849,-0.141497,0.245167,0.257567,1.068394,-0.517027
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YT,0.101486,-0.938713,0.601329,-0.716762,-0.732927,0.905801,0.277137,-0.232287,-0.184810,0.941080,...,-0.311910,-0.527870,-0.815609,-0.773040,-0.646534,0.038814,1.635681,-0.691811,-0.324970,-0.947531
ZR-75-30,-0.709571,0.003607,0.286217,0.785522,0.901308,1.414813,-1.023000,-0.159496,0.142271,0.744544,...,0.108158,1.101814,-0.787272,-0.880678,-1.000956,-0.492708,1.367998,-0.263501,0.564875,-0.438842
huH-1,-0.117474,1.004906,0.093300,0.996833,-0.964825,0.196155,-1.023598,0.120982,-0.477216,1.075433,...,-0.212249,-0.857101,-0.907052,0.217014,0.443357,0.714621,-0.062308,-1.071092,0.112421,0.898853
no-10,-1.014404,0.789371,-0.104136,-0.067664,-1.010729,0.187608,1.756318,-0.110524,0.228141,0.186701,...,0.370875,-0.903403,-0.150268,0.478554,0.059413,-0.267540,1.388255,-0.807648,-0.837319,-0.783107


# Make matrices association matrices by setting 0 threshold and min max scaler.

In [33]:
def min_max_scale(data):
    scaler = MinMaxScaler(feature_range=(0, 1))
    data = data[data > 0].fillna(0)
    return pd.DataFrame(
        scaler.fit_transform(data), index=data.index, columns=data.columns
    )

In [34]:
A_dc = min_max_scale(drug_response)
A_dc

,22RV1,23132-87,42-MG-BA,451Lu,5637,639-V,647-V,697,769-P,786-0,...,WSU-DLCL2,WSU-NHL,YAPC,YH-13,YKG-1,YT,ZR-75-30,huH-1,no-10,no-11
5-Fluorouracil,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5-azacytidine,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.021271,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A-366,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ABT737,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.295231,0.0,0.0,...,0.140539,0.012847,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AGI-5198,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZM447439,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.091113,0.0,0.0,...,0.000000,0.062058,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Zoledronate,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
alpha-lipoic acid,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ascorbate (vitamin C),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [35]:
A_cg = min_max_scale(gene_norm_gene + gene_norm_cell)
A_cg

Gene,A2M,AAED1,ABCC1,ABCC3,ABCG1,ABHD17C,ABI3BP,ABL1,ABL2,ABLIM1,...,ZNF462,ZNF467,ZNF608,ZNF667-AS1,ZNF711,ZNF726,ZNF83,ZNF853,ZP3,ZSCAN31
22RV1,0.100459,0.005994,0.000000,0.000000,0.000000,0.272647,0.000000,0.000000,0.000000,0.482194,...,0.000000,0.00000,0.000000,0.362222,0.000000,0.792077,0.574194,0.000000,0.238351,0.000000
23132-87,0.000000,0.299657,0.000000,0.643529,0.732748,0.689610,0.000000,0.000000,0.000000,0.598057,...,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.314891,0.000000,0.347110,0.000000
42-MG-BA,0.000000,0.217993,0.000000,0.000000,0.000000,0.000000,0.000000,0.242389,0.000000,0.000000,...,0.306933,0.00000,0.227462,0.716783,0.188426,0.000000,0.451683,0.000000,0.000000,0.734761
451Lu,0.713492,0.000000,0.000000,0.000000,0.000000,0.204820,0.000000,0.000000,0.229928,0.000000,...,0.201140,0.00000,0.178506,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5637,0.000000,0.000000,0.011794,0.076162,0.000000,0.078711,0.000000,0.387966,0.032661,0.134330,...,0.736197,0.00000,0.121912,0.000000,0.000000,0.077734,0.000000,0.405834,0.584490,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YT,0.086875,0.000000,0.124164,0.000000,0.000000,0.216302,0.206740,0.000000,0.000000,0.224451,...,0.000000,0.00000,0.000000,0.000000,0.000000,0.026490,0.581660,0.000000,0.000000,0.000000
ZR-75-30,0.000000,0.000000,0.000000,0.533206,0.659150,0.550712,0.000000,0.000000,0.097009,0.185365,...,0.000000,0.55958,0.000000,0.000000,0.000000,0.000000,0.489026,0.009235,0.245194,0.000000
huH-1,0.058726,0.507989,0.000000,0.678837,0.000000,0.000000,0.000000,0.094409,0.000000,0.343345,...,0.000000,0.00000,0.000000,0.141863,0.391946,0.515489,0.000000,0.000000,0.042581,0.569010
no-10,0.000000,0.382332,0.000000,0.093748,0.000000,0.000000,0.800381,0.000000,0.269340,0.000000,...,0.118247,0.00000,0.042781,0.273403,0.198205,0.000000,0.554432,0.000000,0.000000,0.000000


In [36]:
A_dg = (
    pd.DataFrame(
        np.ones([len(A_dc.index), len(A_cg.columns)]),
        index=A_dc.index,
        columns=A_cg.columns,
    )
    / 2
)
for _, i in dti.iterrows():
    A_dg.loc[i["Drug Name"], i["Gene"]] = 1
A_dg

Gene,A2M,AAED1,ABCC1,ABCC3,ABCG1,ABHD17C,ABI3BP,ABL1,ABL2,ABLIM1,...,ZNF462,ZNF467,ZNF608,ZNF667-AS1,ZNF711,ZNF726,ZNF83,ZNF853,ZP3,ZSCAN31
5-Fluorouracil,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
5-azacytidine,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
A-366,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
ABT737,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
AGI-5198,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZM447439,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Zoledronate,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
alpha-lipoic acid,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
ascorbate (vitamin C),0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5


In [37]:
drug_response

,22RV1,23132-87,42-MG-BA,451Lu,5637,639-V,647-V,697,769-P,786-0,...,WSU-DLCL2,WSU-NHL,YAPC,YH-13,YKG-1,YT,ZR-75-30,huH-1,no-10,no-11
5-Fluorouracil,-1.007383,-0.784918,-1.504194,0.000000,-1.354900,-1.987515,-2.477538,-0.978509,-1.361415,-1.754028,...,-1.570807,-0.654756,-2.815973,-2.167938,-1.897658,-0.395993,-3.121705,-2.718147,0.000000,-2.354397
5-azacytidine,-1.386340,-0.882578,-1.740183,0.000000,-1.642768,-0.769647,-1.478863,0.076926,-1.204752,-1.175576,...,-0.196915,-0.318916,-1.863731,-1.153293,0.000000,-0.719780,-2.407082,-1.032133,0.000000,0.000000
A-366,-2.034810,-1.723082,-2.213116,0.000000,-2.101929,-2.346732,-2.175772,-1.446605,-1.556993,-2.153308,...,-0.698646,-1.329787,-2.031633,-1.847300,0.000000,0.000000,-2.828028,-2.721790,0.000000,0.000000
ABT737,-1.731862,-1.285725,-1.311469,-1.176737,-1.118458,-1.923063,-1.495390,1.067689,-0.742262,-1.414749,...,0.402131,0.039985,-2.132064,-1.553697,-1.613575,-1.228735,-1.624036,-1.200161,-1.349079,-1.262084
AGI-5198,-2.716713,-2.175316,-2.030678,-2.144917,-1.930115,-1.715597,-2.176160,-1.681225,-2.522207,-1.719376,...,-1.360011,-1.364146,-2.942588,-2.093594,-1.516215,-1.454616,-2.787429,-2.501823,-2.208319,-2.704476
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZM447439,-2.461575,-1.853879,-0.752992,-2.146206,-0.422651,-1.395663,-0.798400,0.329506,-1.827889,-0.986953,...,-0.552339,0.193145,-2.743585,-1.736393,-0.726710,-0.566273,-1.962101,-0.955661,-1.499584,-1.896599
Zoledronate,-1.787369,-1.517874,-1.544003,0.000000,-1.748420,-1.364055,-1.718075,-1.009534,-1.717543,0.000000,...,0.000000,-1.333803,-2.311593,-1.774683,0.000000,0.000000,-2.227486,-1.900266,0.000000,0.000000
alpha-lipoic acid,-3.448798,-3.495767,-3.460486,0.000000,-3.172817,-3.338942,-3.288135,-2.957743,-3.448755,-3.419779,...,-2.616572,-3.143894,-3.982522,-3.380398,0.000000,-3.137593,-3.831928,-3.665569,0.000000,0.000000
ascorbate (vitamin C),-5.195668,-4.109346,-4.177796,0.000000,-4.133578,-4.659277,-4.043361,-3.548887,-4.462722,-4.266191,...,-2.947327,-4.362139,-5.091152,-4.231438,0.000000,-3.915763,-5.227466,-4.875701,0.000000,0.000000


In [38]:
print(
    "Drug Density: ",
    round(len(A_dc.values.nonzero()[0]) / drug_response.size, 4) * 100,
    "%",
)
print("Cell Density: ", round(len(A_cg.values.nonzero()[0]) / A_cg.size, 4) * 100, "%")
print("Gene Density: ", round(len(A_dg.values.nonzero()[0]) / A_dg.size, 5) * 100, "%")

Drug Density:  11.75 %
Cell Density:  41.949999999999996 %
Gene Density:  100.0 %


# Similarity

In [39]:
def normalize_similarity_matrix(df, gamma=None):
    similarity_matrix = rbf_kernel(df.values, gamma=gamma)
    scaler = MinMaxScaler()
    normalized_matrix = scaler.fit_transform(similarity_matrix.reshape(-1, 1))
    normalized_df = pd.DataFrame(
        normalized_matrix.reshape(similarity_matrix.shape),
        index=df.index,
        columns=df.index,
    )

    return normalized_df

In [40]:
cell_sim = normalize_similarity_matrix(drug_response.T)
cell_sim.to_csv("../GDSC2_data/cell_sim.csv")
cell_sim

,22RV1,23132-87,42-MG-BA,451Lu,5637,639-V,647-V,697,769-P,786-0,...,WSU-DLCL2,WSU-NHL,YAPC,YH-13,YKG-1,YT,ZR-75-30,huH-1,no-10,no-11
22RV1,1.000000,0.639053,0.524479,0.152936,0.547576,0.525271,0.599091,0.240931,0.581277,0.487300,...,0.158288,0.240762,0.376497,0.674361,0.212008,0.236998,0.233196,0.518560,0.218083,0.192309
23132-87,0.639053,1.000000,0.661991,0.193912,0.680823,0.620423,0.671199,0.362530,0.636102,0.633958,...,0.260791,0.338828,0.269753,0.698053,0.270833,0.306838,0.168306,0.543293,0.258440,0.202740
42-MG-BA,0.524479,0.661991,1.000000,0.185470,0.659805,0.593831,0.602424,0.405227,0.625931,0.578249,...,0.240648,0.364568,0.228713,0.627858,0.279524,0.310421,0.119388,0.441735,0.259480,0.193384
451Lu,0.152936,0.193912,0.185470,1.000000,0.175777,0.171185,0.177671,0.112401,0.205716,0.175670,...,0.156333,0.109491,0.077724,0.172249,0.547566,0.179325,0.058856,0.141583,0.555635,0.494428
5637,0.547576,0.680823,0.659805,0.175777,1.000000,0.743042,0.691143,0.307986,0.728520,0.619843,...,0.176644,0.261731,0.347058,0.691983,0.248318,0.245618,0.190474,0.570788,0.223369,0.172327
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YT,0.236998,0.306838,0.310421,0.179325,0.245618,0.278759,0.261869,0.413655,0.298745,0.252880,...,0.593181,0.398291,0.054474,0.277348,0.228450,1.000000,0.029171,0.179992,0.251151,0.126075
ZR-75-30,0.233196,0.168306,0.119388,0.058856,0.190474,0.159988,0.185964,0.017055,0.196267,0.145342,...,0.018563,0.019427,0.480828,0.190951,0.069741,0.029171,1.000000,0.295250,0.074079,0.107138
huH-1,0.518560,0.543293,0.441735,0.141583,0.570788,0.533694,0.555938,0.160023,0.539669,0.473534,...,0.124659,0.148278,0.422588,0.539865,0.195504,0.179992,0.295250,1.000000,0.185867,0.176295
no-10,0.218083,0.258440,0.259480,0.555635,0.223369,0.240579,0.248479,0.166652,0.251032,0.256297,...,0.237944,0.160280,0.093903,0.256889,0.725453,0.251151,0.074079,0.185867,1.000000,0.637483


In [41]:
print("Min:", np.min(cell_sim.values))
print("Max:", np.max(cell_sim.values))
print("Mean:", np.mean(cell_sim.values))
print("Median:", np.median(cell_sim.values))

Min: 0.0
Max: 1.0
Mean: 0.33588797187358
Median: 0.3084658203765811


In [42]:
gene_sim = normalize_similarity_matrix(gene_norm_cell.T)
gene_sim.to_csv("../GDSC2_data/gene_sim.csv")
gene_sim

Gene,A2M,AAED1,ABCC1,ABCC3,ABCG1,ABHD17C,ABI3BP,ABL1,ABL2,ABLIM1,...,ZNF462,ZNF467,ZNF608,ZNF667-AS1,ZNF711,ZNF726,ZNF83,ZNF853,ZP3,ZSCAN31
Gene,,,,,,,,,,,,,,,,,,,,,
A2M,1.000000,0.124329,0.078252,0.089120,0.073464,0.106634,0.123678,0.117898,0.182402,0.081458,...,0.141055,0.092254,0.159685,0.132719,0.083785,0.115042,0.098818,0.130634,0.079839,0.089856
AAED1,0.124329,1.000000,0.107053,0.167308,0.065102,0.141813,0.181372,0.176025,0.218780,0.084843,...,0.210526,0.112547,0.079851,0.129800,0.059145,0.064622,0.097590,0.098259,0.172437,0.152982
ABCC1,0.078252,0.107053,1.000000,0.222812,0.132680,0.122020,0.131405,0.144803,0.111102,0.166865,...,0.138764,0.115446,0.115049,0.098834,0.096714,0.090184,0.090591,0.097735,0.129370,0.169278
ABCC3,0.089120,0.167308,0.222812,1.000000,0.132498,0.248266,0.162946,0.108702,0.111524,0.247987,...,0.154924,0.125198,0.095233,0.076690,0.047717,0.052685,0.104071,0.084652,0.191703,0.228976
ABCG1,0.073464,0.065102,0.132680,0.132498,1.000000,0.156158,0.080018,0.092892,0.057030,0.187331,...,0.079098,0.172672,0.098011,0.083655,0.167224,0.133834,0.113995,0.132175,0.118674,0.146407
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF726,0.115042,0.064622,0.090184,0.052685,0.133834,0.053628,0.089016,0.115036,0.088017,0.085601,...,0.098543,0.119580,0.155175,0.208555,0.229802,1.000000,0.188119,0.148970,0.072373,0.097496
ZNF83,0.098818,0.097590,0.090591,0.104071,0.113995,0.069678,0.143087,0.159338,0.120779,0.089213,...,0.119280,0.161838,0.170793,0.214104,0.142774,0.188119,1.000000,0.153549,0.097066,0.109650
ZNF853,0.130634,0.098259,0.097735,0.084652,0.132175,0.099513,0.093910,0.152894,0.112496,0.125016,...,0.196814,0.172694,0.181429,0.252914,0.269150,0.148970,0.153549,1.000000,0.080650,0.122931


In [43]:
print("Min:", np.min(gene_sim.values))
print("Max:", np.max(gene_sim.values))
print("Mean:", np.mean(gene_sim.values))
print("Median:", np.median(gene_sim.values))

Min: 0.0
Max: 1.0
Mean: 0.13364979286993234
Median: 0.11899529690948964


# NSC to SMILES

In [44]:
def get_morgan_fingerprint(SMILES):
    # Initialize parser parameters
    params = Chem.SmilesParserParams()
    params.useChirality = True  # Preserve stereochemistry information
    params.removeHs = False  # Keep hydrogen atoms
    mfp = []

    for smi in SMILES:
        mol = None
        # Sanitization attempt strategies
        sanitize_attempts = [
            {"sanitize": True},  # First try with standard sanitization
            {
                "sanitize": False,
                "partial_sanitize": True,
            },  # Fallback: partial sanitization
        ]

        for attempt in sanitize_attempts:
            try:
                # Update parameters for this attempt
                current_params = Chem.SmilesParserParams()
                current_params.sanitize = attempt["sanitize"]
                current_params.useChirality = params.useChirality
                current_params.removeHs = params.removeHs

                # Molecule object creation
                mol = Chem.MolFromSmiles(smi, params=current_params)

                if mol and "partial_sanitize" in attempt:
                    # Perform partial sanitization (skip property validation)
                    Chem.SanitizeMol(mol, Chem.SANITIZE_ALL ^ Chem.SANITIZE_PROPERTIES)

                if mol:  # Successfully processed molecule
                    # Generate Morgan fingerprint
                    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
                    mfp.append(np.array(fp))
                    break  # Exit attempt loop on success

            except Exception as e:
                if attempt == sanitize_attempts[-1]:  # Final attempt failed
                    print(f"Processing failed: {smi}")
                    print(f"Error details: {str(e)}")
                continue  # Try next attempt

        if not mol:  # All attempts failed
            print(f"Complete processing failure: {smi}")
            mfp.append(np.zeros(2048))  # Insert zero-vector placeholder

    return np.array(mfp)

In [45]:
conv = dict(SMILES[["drugs", "SMILES"]].values)
mfp = get_morgan_fingerprint([conv[i] for i in drug_response.index])
mfp = pd.DataFrame(mfp, index=drug_response.index)

In [46]:
drug_sim = normalize_similarity_matrix(mfp)
drug_sim.to_csv("../GDSC2_data/drug_sim.csv")
drug_sim

,5-Fluorouracil,5-azacytidine,A-366,ABT737,AGI-5198,AGI-6780,AGK2,AMG-319,AT13148,AZ6102,...,WZ4003,Wee1 Inhibitor,Wnt-C59,XAV939,YK-4-279,ZM447439,Zoledronate,alpha-lipoic acid,ascorbate (vitamin C),glutathione
5-Fluorouracil,1.000000,0.759035,0.694739,0.504284,0.640653,0.655375,0.640653,0.684884,0.768963,0.699671,...,0.596617,0.754074,0.709540,0.793825,0.754074,0.616165,0.798804,0.778900,0.818748,0.739208
5-azacytidine,0.759035,1.000000,0.665202,0.456031,0.611274,0.577107,0.591736,0.596617,0.699671,0.640653,...,0.577107,0.665202,0.660287,0.675038,0.675038,0.586857,0.749116,0.719420,0.768963,0.689810
A-366,0.694739,0.665202,1.000000,0.431992,0.557635,0.533349,0.528499,0.552773,0.655375,0.586857,...,0.591736,0.601500,0.625953,0.621058,0.660287,0.699671,0.675038,0.665202,0.665202,0.635751
ABT737,0.504284,0.456031,0.431992,1.000000,0.417597,0.480128,0.436795,0.393652,0.523651,0.456031,...,0.470482,0.499448,0.475304,0.460845,0.489783,0.480128,0.494614,0.494614,0.523651,0.494614
AGI-5198,0.640653,0.611274,0.557635,0.417597,1.000000,0.567366,0.523651,0.596617,0.572236,0.552773,...,0.518806,0.557635,0.611274,0.557635,0.577107,0.538202,0.630851,0.621058,0.640653,0.591736
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZM447439,0.616165,0.586857,0.699671,0.480128,0.538202,0.513963,0.547914,0.572236,0.586857,0.528499,...,0.611274,0.562500,0.625953,0.562500,0.601500,1.000000,0.625953,0.616165,0.635751,0.596617
Zoledronate,0.798804,0.749116,0.675038,0.494614,0.630851,0.616165,0.630851,0.616165,0.749116,0.640653,...,0.586857,0.684884,0.729309,0.704604,0.704604,0.625953,1.000000,0.759035,0.778900,0.719420
alpha-lipoic acid,0.778900,0.719420,0.665202,0.494614,0.621058,0.645558,0.601500,0.586857,0.709540,0.621058,...,0.586857,0.675038,0.679960,0.704604,0.704604,0.616165,0.759035,1.000000,0.778900,0.768963
ascorbate (vitamin C),0.818748,0.768963,0.665202,0.523651,0.640653,0.635751,0.630851,0.645558,0.729309,0.660287,...,0.596617,0.724363,0.699671,0.714479,0.724363,0.635751,0.778900,0.778900,1.000000,0.739208


In [47]:
print("Min:", np.min(drug_sim.values))
print("Max:", np.max(drug_sim.values))
print("Mean:", np.mean(drug_sim.values))
print("Median:", np.median(drug_sim.values))

Min: 0.0
Max: 1.0
Mean: 0.5638407152074424
Median: 0.5722355734328417


# Unified Graph

In [48]:
indexes = list(A_dc.index) + list(A_cg.index) + list(A_dg.columns)
n_all = len(indexes)
base = pd.DataFrame(np.zeros([n_all, n_all]), index=indexes, columns=indexes)
base

,5-Fluorouracil,5-azacytidine,A-366,ABT737,AGI-5198,AGI-6780,AGK2,AMG-319,AT13148,AZ6102,...,ZNF462,ZNF467,ZNF608,ZNF667-AS1,ZNF711,ZNF726,ZNF83,ZNF853,ZP3,ZSCAN31
5-Fluorouracil,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5-azacytidine,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A-366,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ABT737,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AGI-5198,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF726,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZNF83,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZNF853,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZP3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [49]:
A_dc

,22RV1,23132-87,42-MG-BA,451Lu,5637,639-V,647-V,697,769-P,786-0,...,WSU-DLCL2,WSU-NHL,YAPC,YH-13,YKG-1,YT,ZR-75-30,huH-1,no-10,no-11
5-Fluorouracil,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5-azacytidine,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.021271,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A-366,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ABT737,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.295231,0.0,0.0,...,0.140539,0.012847,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AGI-5198,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZM447439,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.091113,0.0,0.0,...,0.000000,0.062058,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Zoledronate,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
alpha-lipoic acid,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ascorbate (vitamin C),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [50]:
A_cg

Gene,A2M,AAED1,ABCC1,ABCC3,ABCG1,ABHD17C,ABI3BP,ABL1,ABL2,ABLIM1,...,ZNF462,ZNF467,ZNF608,ZNF667-AS1,ZNF711,ZNF726,ZNF83,ZNF853,ZP3,ZSCAN31
22RV1,0.100459,0.005994,0.000000,0.000000,0.000000,0.272647,0.000000,0.000000,0.000000,0.482194,...,0.000000,0.00000,0.000000,0.362222,0.000000,0.792077,0.574194,0.000000,0.238351,0.000000
23132-87,0.000000,0.299657,0.000000,0.643529,0.732748,0.689610,0.000000,0.000000,0.000000,0.598057,...,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.314891,0.000000,0.347110,0.000000
42-MG-BA,0.000000,0.217993,0.000000,0.000000,0.000000,0.000000,0.000000,0.242389,0.000000,0.000000,...,0.306933,0.00000,0.227462,0.716783,0.188426,0.000000,0.451683,0.000000,0.000000,0.734761
451Lu,0.713492,0.000000,0.000000,0.000000,0.000000,0.204820,0.000000,0.000000,0.229928,0.000000,...,0.201140,0.00000,0.178506,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5637,0.000000,0.000000,0.011794,0.076162,0.000000,0.078711,0.000000,0.387966,0.032661,0.134330,...,0.736197,0.00000,0.121912,0.000000,0.000000,0.077734,0.000000,0.405834,0.584490,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YT,0.086875,0.000000,0.124164,0.000000,0.000000,0.216302,0.206740,0.000000,0.000000,0.224451,...,0.000000,0.00000,0.000000,0.000000,0.000000,0.026490,0.581660,0.000000,0.000000,0.000000
ZR-75-30,0.000000,0.000000,0.000000,0.533206,0.659150,0.550712,0.000000,0.000000,0.097009,0.185365,...,0.000000,0.55958,0.000000,0.000000,0.000000,0.000000,0.489026,0.009235,0.245194,0.000000
huH-1,0.058726,0.507989,0.000000,0.678837,0.000000,0.000000,0.000000,0.094409,0.000000,0.343345,...,0.000000,0.00000,0.000000,0.141863,0.391946,0.515489,0.000000,0.000000,0.042581,0.569010
no-10,0.000000,0.382332,0.000000,0.093748,0.000000,0.000000,0.800381,0.000000,0.269340,0.000000,...,0.118247,0.00000,0.042781,0.273403,0.198205,0.000000,0.554432,0.000000,0.000000,0.000000


In [51]:
A_dg

Gene,A2M,AAED1,ABCC1,ABCC3,ABCG1,ABHD17C,ABI3BP,ABL1,ABL2,ABLIM1,...,ZNF462,ZNF467,ZNF608,ZNF667-AS1,ZNF711,ZNF726,ZNF83,ZNF853,ZP3,ZSCAN31
5-Fluorouracil,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
5-azacytidine,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
A-366,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
ABT737,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
AGI-5198,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZM447439,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
Zoledronate,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
alpha-lipoic acid,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
ascorbate (vitamin C),0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5


In [52]:
base.loc[A_cg.index, A_cg.columns] = A_cg
base.loc[A_cg.columns, A_cg.index] = A_cg.T
base.loc[A_dc.index, A_dc.columns] = A_dc
base.loc[A_dc.columns, A_dc.index] = A_dc.T
base.loc[A_dg.index, A_dg.columns] = A_dg
base.loc[A_dg.columns, A_dg.index] = A_dg.T

In [53]:
edge_index = np.array(base.values.nonzero())
edge_attr = np.array(base.values[base.values.nonzero()])

np.save(
    "../GDSC2_data/idxs.npy",
    pd.DataFrame([list(range(len(base.index))), base.index]).values,
)

np.save(
    "../GDSC2_data/edge_idxs.npy",
    edge_index,
)

np.save(
    "../GDSC2_data/edge_attr.npy",
    edge_attr,
)